Cell 1: Imports & Setup

In [21]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

import tensorflow as tf
from tensorflow.keras import layers, Model, Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import KLDivergence


# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

sns.set_theme(style="whitegrid")
FEATURE_MATRIX_PATH = "D:\Project\Customer-segmentation\data\cleanCustomerFeature.csv"

<>:22: SyntaxWarning: invalid escape sequence '\P'
<>:22: SyntaxWarning: invalid escape sequence '\P'
C:\Users\ASUS\AppData\Local\Temp\ipykernel_30376\1810357339.py:22: SyntaxWarning: invalid escape sequence '\P'
  FEATURE_MATRIX_PATH = "D:\Project\Customer-segmentation\data\cleanCustomerFeature.csv"


Cell 2: Load Data & Standardize Features

In [22]:
# Load feature matrix from Notebook 02
df = pd.read_csv(FEATURE_MATRIX_PATH)

# Select log-transformed behavioral features
log_feature_cols = [col for col in df.columns if col.endswith('_log')]
X_raw = df[log_feature_cols].values

# Standardize features (Mean=0, Std=1) for neural networks & distance-based clustering
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print(f"Dataset ready: {X_scaled.shape[0]} customers across {X_scaled.shape[1]} log-scaled features.")
print(log_feature_cols)

Dataset ready: 4332 customers across 7 log-scaled features.
['Recency_log', 'Frequency_log', 'Monetary_log', 'TotalQuantity_log', 'TotalItems_log', 'AvgOrderValue_log', 'AvgInterpurchaseDays_log']


Cell 3: Evaluation Metrics Utility Function

In [23]:
def evaluate_clustering(model_name, X_space, labels):
    """Computes Silhouette Score, Davies-Bouldin Index, and Calinski-Harabasz Index."""
    n_clusters = len(np.unique(labels))
    if n_clusters < 2:
        return None
    
    sil = silhouette_score(X_space, labels)
    db = davies_bouldin_score(X_space, labels)
    ch = calinski_harabasz_score(X_space, labels)
    
    return {
        "Model": model_name,
        "Clusters (K)": n_clusters,
        "Silhouette Score (↑)": round(sil, 4),
        "Davies-Bouldin Index (↓)": round(db, 4),
        "Calinski-Harabasz Index (↑)": round(ch, 2)
    }

results_list = []

Cell 4: Model 1 — Traditional Baseline: K-Means on Raw Features

In [24]:
# K-Means directly on standardized features (as cited in literature [3, 5, 10])
for k in [3, 4, 5]:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    res = evaluate_clustering(f"Baseline: K-Means (K={k})", X_scaled, labels)
    results_list.append(res)

pd.DataFrame(results_list)

,Model,Clusters (K),Silhouette Score (↑),Davies-Bouldin Index (↓),Calinski-Harabasz Index (↑)
0,Baseline: K-Means (K=3),3,0.3080,1.1852,2575.33
1,Baseline: K-Means (K=4),4,0.3139,1.1867,2088.46
2,Baseline: K-Means (K=5),5,0.2729,1.2552,1875.55


Cell 5: Model 2 — Linear Dimensionality Reduction: PCA + K-Means

In [25]:
# Linear compression to 3 dimensions before clustering
pca = PCA(n_components=3, random_state=42)
X_pca = pca.fit_transform(X_scaled)

for k in [3, 4, 5]:
    kmeans_pca = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_pca = kmeans_pca.fit_predict(X_pca)
    res = evaluate_clustering(f"PCA + K-Means (K={k})", X_pca, labels_pca)
    results_list.append(res)

pd.DataFrame(results_list)

,Model,Clusters (K),Silhouette Score (↑),Davies-Bouldin Index (↓),Calinski-Harabasz Index (↑)
0,Baseline: K-Means (K=3),3,0.3080,1.1852,2575.33
1,Baseline: K-Means (K=4),4,0.3139,1.1867,2088.46
2,Baseline: K-Means (K=5),5,0.2729,1.2552,1875.55
3,PCA + K-Means (K=3),3,0.3309,1.0649,3256.16
4,PCA + K-Means (K=4),4,0.3422,1.0220,2759.19
5,PCA + K-Means (K=5),5,0.3040,1.0933,2513.68


Cell 6: Model 3 — Autoencoder Latent Embeddings + K-Means (Proposed Framework)

In [26]:
input_dim = X_scaled.shape[1]
latent_dim = 3  # Bottleneck embedding size

# Build Encoder
encoder_input = layers.Input(shape=(input_dim,), name="Encoder_Input")
x = layers.Dense(16, activation="relu")(encoder_input)
x = layers.BatchNormalization()(x)
x = layers.Dense(8, activation="relu")(x)
latent_bottleneck = layers.Dense(latent_dim, activation="linear", name="Latent_Space")(x)

encoder = Model(inputs=encoder_input, outputs=latent_bottleneck, name="Encoder")

# Build Decoder
decoder_input = layers.Input(shape=(latent_dim,), name="Decoder_Input")
y = layers.Dense(8, activation="relu")(decoder_input)
y = layers.Dense(16, activation="relu")(y)
reconstructed_output = layers.Dense(input_dim, activation="linear")(y)

decoder = Model(inputs=decoder_input, outputs=reconstructed_output, name="Decoder")

# Build Full Autoencoder
autoencoder_output = decoder(encoder(encoder_input))
autoencoder = Model(inputs=encoder_input, outputs=autoencoder_output, name="Autoencoder")

autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss="mse")

# Train Autoencoder
history = autoencoder.fit(
    X_scaled, X_scaled,
    epochs=100,
    batch_size=32,
    validation_split=0.15,
    verbose=0
)

# Extract Latent Customer Embeddings
embeddings = encoder.predict(X_scaled)

# Perform K-Means on Latent Space
for k in [3, 4, 5]:
    kmeans_ae = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_ae = kmeans_ae.fit_predict(embeddings)
    res = evaluate_clustering(f"Autoencoder + K-Means (K={k})", embeddings, labels_ae)
    results_list.append(res)

pd.DataFrame(results_list)

136/136 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


,Model,Clusters (K),Silhouette Score (↑),Davies-Bouldin Index (↓),Calinski-Harabasz Index (↑)
0,Baseline: K-Means (K=3),3,0.3080,1.1852,2575.33
1,Baseline: K-Means (K=4),4,0.3139,1.1867,2088.46
2,Baseline: K-Means (K=5),5,0.2729,1.2552,1875.55
3,PCA + K-Means (K=3),3,0.3309,1.0649,3256.16
4,PCA + K-Means (K=4),4,0.3422,1.0220,2759.19
5,PCA + K-Means (K=5),5,0.3040,1.0933,2513.68
6,Autoencoder + K-Means (K=3),3,0.4168,0.9453,5496.97
7,Autoencoder + K-Means (K=4),4,0.3426,0.9902,5239.55
8,Autoencoder + K-Means (K=5),5,0.3582,0.9483,5049.86


Cell 7: Model 4 — Deep Embedded Clustering (DEC) Refinement

In [27]:

class ClusteringLayer(layers.Layer):
    """Custom clustering layer that computes Student's t-distribution kernel."""
    def __init__(self, n_clusters=4, weights=None, alpha=1.0, **kwargs):
        super(ClusteringLayer, self).__init__(**kwargs)
        self.n_clusters = n_clusters
        self.alpha = alpha
        self.initial_weights = weights

    def build(self, input_shape):
        self.clusters = self.add_weight(
            shape=(self.n_clusters, input_shape[1]),
            initializer="glorot_uniform",
            name="cluster_centers"
        )
        if self.initial_weights is not None:
            self.set_weights(self.initial_weights)
        self.built = True

    def call(self, inputs):
        # q_ij = (1 + ||z_i - u_j||^2 / alpha) ^ -((alpha+1)/2)
        q = 1.0 / (1.0 + (tf.reduce_sum(tf.square(tf.expand_dims(inputs, axis=1) - self.clusters), axis=2) / self.alpha))
        q **= (self.alpha + 1.0) / 2.0
        q = tf.transpose(tf.transpose(q) / tf.reduce_sum(q, axis=1))
        return q

# Initialize DEC with K=3
k_opt = (3,4,5)
for k in k_opt:
    kmeans_init = KMeans(n_clusters=k, random_state=42, n_init=10)
    init_labels = kmeans_init.fit_predict(embeddings)
    cluster_centers_init = kmeans_init.cluster_centers_

    # Construct DEC Model
    clustering_output = ClusteringLayer(n_clusters=k, weights=[cluster_centers_init], name="DEC_Layer")(encoder.output)
    dec_model = Model(inputs=encoder.input, outputs=clustering_output)

    # Fix: Use KLDivergence() object directly to avoid version identifier string issues
    dec_model.compile(optimizer=Adam(learning_rate=0.0001), loss=KLDivergence())

    # DEC Target Distribution Function
    def target_distribution(q):
        weight = q ** 2 / q.sum(0)
        return (weight.T / weight.sum(1)).T

    # DEC Fine-Tuning Loop
    q = dec_model.predict(X_scaled, verbose=0)
    p = target_distribution(q)
    dec_model.fit(X_scaled, p, epochs=30, batch_size=32, verbose=0)

    # Extract DEC Clusters
    q_final = dec_model.predict(X_scaled, verbose=0)
    dec_labels = q_final.argmax(axis=1)

    res = evaluate_clustering(f"DEC (K={k})", embeddings, dec_labels)
    results_list.append(res)

print("DEC Model Trained & Evaluated Successfully!")

DEC Model Trained & Evaluated Successfully!


Cell 8: Consolidate & Export Benchmark Results Table

In [28]:
# Convert to DataFrame
benchmark_df = pd.DataFrame(results_list)
print("=== MODEL COMPARISON RESULTS ===")
display(benchmark_df)

# Save results for paper / presentation
benchmark_df.to_csv("D:\\Project\\Customer-segmentation\\results.csv", index=False)

=== MODEL COMPARISON RESULTS ===


,Model,Clusters (K),Silhouette Score (↑),Davies-Bouldin Index (↓),Calinski-Harabasz Index (↑)
0,Baseline: K-Means (K=3),3,0.3080,1.1852,2575.33
1,Baseline: K-Means (K=4),4,0.3139,1.1867,2088.46
2,Baseline: K-Means (K=5),5,0.2729,1.2552,1875.55
3,PCA + K-Means (K=3),3,0.3309,1.0649,3256.16
4,PCA + K-Means (K=4),4,0.3422,1.0220,2759.19
5,PCA + K-Means (K=5),5,0.3040,1.0933,2513.68
6,Autoencoder + K-Means (K=3),3,0.4168,0.9453,5496.97
7,Autoencoder + K-Means (K=4),4,0.3426,0.9902,5239.55
8,Autoencoder + K-Means (K=5),5,0.3582,0.9483,5049.86
9,DEC (K=3),3,0.4106,0.9582,5458.50


Autoencoder + k-means(k=3) performs the best

Cell 9: Assign Final Cluster Labels & Export Profiled Customers

In [29]:
OPTIMAL_K = 3
# 1. Fit K-Means with K=3 on Autoencoder Latent Embeddings
best_kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
final_cluster_labels = best_kmeans.fit_predict(embeddings)

# 2. Assign cluster labels back to the DataFrame
df['Cluster'] = final_cluster_labels

# 3. Identify available RFM columns (handles both raw and log-transformed column names safely)
rfm_cols = {}
for col in ['Recency', 'Frequency', 'Monetary', 'AvgOrderValue']:
    if col in df.columns:
        rfm_cols[col] = 'mean'
    elif f'{col}_log' in df.columns:
        rfm_cols[f'{col}_log'] = 'mean'

# 4. Generate Cluster Profile Summary
cluster_summary = df.groupby('Cluster').agg(rfm_cols).round(2)
cluster_summary['Customer_Count'] = df.groupby('Cluster').size()

print("=== SEGMENT PERSONA PROFILE SUMMARY (K=4) ===")
print(cluster_summary)

output_path = "D:\\Project\\Customer-segmentation\\segmented_customers.csv"
df.to_csv(output_path, index=False)
print(f"\nFinal customer segmentations exported successfully to: {output_path}")

=== SEGMENT PERSONA PROFILE SUMMARY (K=4) ===
         Recency  Frequency  Monetary  AvgOrderValue  Customer_Count
Cluster                                                             
0          80.32       1.63    583.18         126.55            1167
1          52.28       6.69   3165.01          89.70            2316
2         220.20       1.04    257.02          57.66             849

Final customer segmentations exported successfully to: D:\Project\Customer-segmentation\segmented_customers.csv


In [30]:
import pickle

# Save embeddings and labels
np.save(r'D:\Project\Customer-segmentation\embeddings.npy', embeddings)
np.save(
    r'D:\Project\Customer-segmentation\final_cluster_labels.npy',
    final_cluster_labels,
)

# Save training history dictionary
with open(
    r'D:\Project\Customer-segmentation\history_dict.pkl', 'wb'
) as file_handle:
  pickle.dump(history.history, file_handle)